# Predicting House Prices with Linear Regression
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 1**

**Objective:** build and evaluate a linear regression model that predicts house prices from features such as area, location, number of rooms and age, covering the full workflow from data cleaning to model interpretation.

**Dataset:** `house_prices.csv` - 1,500 houses with 8 features (Area_sqft, Bedrooms, Bathrooms, Age_years, Parking, Location, Furnishing, Main_Road) and the target `Price` in Rs. lakh.

> The file included here is generated by `generate_data.py` (realistic features with some missing values). To use a real dataset such as Kaggle's *House Prices: Advanced Regression Techniques* or *Housing Prices*, save it as `house_prices.csv` with a numeric `Price` column, adapt the feature lists in section 4, and update the written observations.

**Plan:** load and EDA -> target distribution -> feature selection discussion -> missing values and encoding -> correlation heatmap -> train/test split (80/20) -> linear regression -> evaluation (MSE, RMSE, R2) -> actual vs predicted -> residuals -> coefficients -> bonus: Ridge and Lasso.

---
## 1. Load the data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

df = pd.read_csv("house_prices.csv")
print("Shape:", df.shape)
df.head()

## 2. Exploratory data analysis: nulls, types and descriptive statistics

In [ ]:
print("Null values per column:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nDescriptive statistics (numeric):")
print(df.describe().round(1))
print("\nCategorical columns:")
for c in ["Location", "Furnishing", "Main_Road"]:
    print(c, df[c].value_counts(dropna=False).to_dict())

**Observation:** the dataset has 1,500 houses and 9 columns. Four columns have missing values: **Age_years 75 (5%)**, **Furnishing 60 (4%)**, **Bathrooms 45 (3%)** and **Location 30 (2%)**; the rest are complete. The average house is about 1,429 sqft (range 450 to 3,981), with 2.3 bedrooms and 1.7 bathrooms. Three columns are text categories (Location, Furnishing, Main_Road) and must be encoded before modelling.

## 3. Distribution of the target variable (price)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df["Price"], kde=True, bins=40, ax=ax[0], color="steelblue")
ax[0].set_title("Distribution of House Price (Rs. lakh)")
sns.boxplot(x=df["Price"], ax=ax[1], color="lightcoral"); ax[1].set_title("Price Boxplot")
plt.tight_layout()
plt.savefig("images/01_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Price skewness:", round(df["Price"].skew(), 2))
print(df["Price"].describe().round(1))
print("\nAverage price by Location:\n", df.groupby("Location")["Price"].mean().round(1).sort_values(ascending=False))

**Observation:** prices average **76.4 lakh** (median 73.5, standard deviation 30.9) and range from 15 to about 201 lakh. The skewness of 0.6 means the distribution is **mildly right-skewed**: most houses are in the 50-100 lakh range, with a tail of expensive properties (visible as outliers in the boxplot). Location clearly matters: the average price in the City Center (101.5 lakh) is almost double that of the Industrial Area (57.1 lakh).

## 4. Feature selection discussion

Which features should predict price, and why?

| Feature | Expected effect | Reasoning |
|---|---|---|
| **Area_sqft** | strong positive | Bigger houses cost more; usually the single most important driver. |
| **Location** | strong (positive or negative) | Neighbourhood determines demand, services and prestige. |
| **Bathrooms / Bedrooms** | positive | More rooms means more usable space. But they are correlated with area and with each other (multicollinearity), so their individual coefficients need careful reading. |
| **Age_years** | negative | Older houses depreciate and need more maintenance. |
| **Parking, Main_Road, Furnishing** | smaller positive | Convenience and amenities add value on top of size and location. |

All 8 features are plausible predictors and the number is small, so we **keep all of them** and let the model weigh them. We check the choice with the correlation heatmap and the coefficients later.

## 5. Handle missing values and encode categorical features
- **Numeric columns:** missing values are filled with the **median** (robust to outliers).
- **Categorical columns:** missing values are filled with the **most frequent value (mode)**.
- **One-Hot Encoding** turns each category into 0/1 columns. We drop the first category of each column to avoid a redundant column (the "dummy variable trap"), which makes that category the **baseline**: City Center for Location, Fully-Furnished for Furnishing and "No" for Main_Road.
- We **split first (80/20) and then learn the fill values from the training set only**, so no information from the test set leaks into the model. This is the same set of steps as the checklist, only in the safer order.

In [ ]:
numeric_cols = ["Area_sqft", "Bedrooms", "Bathrooms", "Age_years", "Parking"]
categorical_cols = ["Location", "Furnishing", "Main_Road"]
X = df[numeric_cols + categorical_cols]
y = df["Price"]

# Split FIRST, then learn the imputation values from the training set only (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))]), categorical_cols),
])
Xtr = pd.DataFrame(preprocess.fit_transform(X_train), columns=preprocess.get_feature_names_out(), index=X_train.index)
Xte = pd.DataFrame(preprocess.transform(X_test), columns=preprocess.get_feature_names_out(), index=X_test.index)
Xtr.columns = [c.split("__")[1] for c in Xtr.columns]; Xte.columns = Xtr.columns

print("\nNulls after preprocessing (train/test):", int(Xtr.isnull().sum().sum()), "/", int(Xte.isnull().sum().sum()))
print("Features after one-hot encoding:", list(Xtr.columns))

**Observation:** the training set has 1,200 houses and the test set 300. After preprocessing there are **no missing values left** and the 8 original features have become **12 numeric features** (5 numeric + 4 Location dummies + 2 Furnishing dummies + 1 Main_Road dummy).

## 6. Correlation heatmap

In [ ]:
corr_df = Xtr.copy(); corr_df["Price"] = y_train
corr = corr_df.corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 8})
plt.title("Correlation Heatmap (training data, after encoding)")
plt.savefig("images/02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Correlation Area_sqft vs Bedrooms:", round(corr.loc["Area_sqft", "Bedrooms"], 2), "| Area_sqft vs Bathrooms:", round(corr.loc["Area_sqft", "Bathrooms"], 2), "| Bedrooms vs Bathrooms:", round(corr.loc["Bedrooms", "Bathrooms"], 2), "\n")
print("Correlation with Price:")
print(corr["Price"].drop("Price").sort_values(ascending=False).round(2))

**Observation:**
- **Area_sqft is by far the most correlated with price (0.82)**, followed by Bedrooms (0.65) and Bathrooms (0.54). On the negative side: Age_years (-0.23), Outskirts (-0.21) and Industrial Area (-0.21); Riverside is positive (+0.22).
- Bedrooms and Bathrooms correlate strongly with area (0.71 and 0.54) and with each other (0.73), so much of their correlation with price comes through house size. This is **multicollinearity**: predictions stay accurate, but their individual coefficients are harder to interpret.
- Parking and Main_Road have weak correlations with price (0.07), but correlation looks at one feature at a time. The regression below shows they still add value once area and location are held constant.

## 7. Train the linear regression model

In [ ]:
model = LinearRegression()
model.fit(Xtr, y_train)
pred_train = pd.Series(model.predict(Xtr), index=Xtr.index)
pred_test = pd.Series(model.predict(Xte), index=Xte.index)

## 8. Evaluate the model: MSE, RMSE and R2

In [ ]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred)}

lr_results = pd.DataFrame({"Train": metrics(y_train, pred_train), "Test": metrics(y_test, pred_test)}).round(3)
print(lr_results)
cv = cross_val_score(Pipeline([("pre", preprocess), ("lr", LinearRegression())]), X, y, cv=5, scoring="r2")
print("\n5-fold cross-validated R2:", cv.round(3), "| mean =", cv.mean().round(3))
print(f"Average price in test set: {y_test.mean():.1f} lakh -> RMSE is {metrics(y_test, pred_test)['RMSE'] / y_test.mean() * 100:.1f}% of the average price")

**Observation:**

| Metric | Train | Test |
|---|---|---|
| MSE | 55.4 | 49.3 |
| RMSE | 7.4 | 7.0 |
| MAE | 5.9 | 5.5 |
| R2 | 0.941 | 0.951 |

- **R2 = 0.951 on the test set:** the model explains about **95% of the variation** in house prices.
- **RMSE = 7.0 lakh:** a typical prediction is off by about 7 lakh, which is **9.2% of the average price** (76.3 lakh).
- Train and test scores are very close (and 5-fold cross-validation gives R2 around 0.94 for every fold), so the model **does not overfit** and the result is not a lucky split. The test R2 being slightly higher than train is just chance in this particular split.

## 9. Scatter plot: actual vs predicted prices

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, pred_test, alpha=0.5, color="teal")
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual price (lakh)"); plt.ylabel("Predicted price (lakh)")
plt.title(f"Actual vs Predicted Prices (test set, R2 = {r2_score(y_test, pred_test):.3f})"); plt.legend()
plt.savefig("images/03_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** the points follow the red diagonal ("perfect prediction" line) closely across the whole price range, from cheap houses (about 20 lakh) to expensive ones (about 180 lakh). There is no visible curve or gap, so a straight-line model describes this data well.

## 10. Residual plot

In [ ]:
residuals = y_test - pred_test
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(pred_test, residuals, alpha=0.5, color="purple"); ax[0].axhline(0, color="red", linestyle="--")
ax[0].set_xlabel("Predicted price (lakh)"); ax[0].set_ylabel("Residual (actual - predicted)"); ax[0].set_title("Residuals vs Predicted")
sns.histplot(residuals, kde=True, bins=30, ax=ax[1], color="purple"); ax[1].set_title("Distribution of Residuals")
plt.tight_layout()
plt.savefig("images/04_residuals.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Residual mean: {residuals.mean():.2f} | std: {residuals.std():.2f} | skew: {residuals.skew():.2f}")
low, high = pred_test < pred_test.median(), pred_test >= pred_test.median()
print(f"Residual spread (std) for cheaper half: {residuals[low].std():.2f} | for pricier half: {residuals[high].std():.2f}")

**Observation:** the residuals (actual minus predicted) are **randomly scattered around zero** with no pattern. Their mean is about 0 (0.12), they form a roughly symmetric bell shape (skewness 0.28), and the spread is similar for cheaper houses (std 7.3) and pricier houses (std 6.7). So the main assumptions of linear regression hold: errors are centred, roughly normal, and have a constant spread. A few houses are mispredicted by more than 20 lakh, but these are rare.

## 11. Coefficient analysis: which features matter most?

In [ ]:
coef = pd.Series(model.coef_, index=Xtr.columns)
print(f"Intercept: {model.intercept_:.2f}\n")
print("Raw coefficients (change in price, lakh, for +1 unit of the feature):")
print(coef.sort_values(ascending=False).round(3))

# Raw coefficients are in different units (sqft vs yes/no), so also compare on a standard scale
std_model = LinearRegression().fit(StandardScaler().fit_transform(Xtr), y_train)
std_coef = pd.Series(std_model.coef_, index=Xtr.columns).sort_values()
print("\nStandardised coefficients (change in price, lakh, for +1 standard deviation):")
print(std_coef.sort_values(ascending=False).round(2))

plt.figure(figsize=(9, 6))
colors = ["#DC2626" if v < 0 else "#16A34A" for v in std_coef.values]
plt.barh(std_coef.index, std_coef.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Impact of each feature on price (standardised coefficients)"); plt.xlabel("Change in price (lakh) per +1 std dev")
plt.savefig("images/05_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

**How to read the coefficients:** a raw coefficient is the change in price (in lakh) when that feature increases by 1 unit and everything else stays the same. For one-hot columns it is the difference **compared with the baseline category** (City Center, Fully-Furnished, Main_Road = No). Because features have different units, the standardised version (per +1 standard deviation) is used to compare their importance fairly.

**Observation:**
- **Highest positive impact:** each extra square foot adds about 0.044 lakh (so +100 sqft is roughly +4.4 lakh). Having a main road adds **+5.2 lakh**, each extra bathroom **+4.6 lakh**, and each parking space **+3.2 lakh**. On the standardised scale, **Area_sqft is the most important feature** (+21.8 lakh per standard deviation, far ahead of the rest).
- **Highest negative impact:** compared with City Center, an **Industrial Area house is 45.2 lakh cheaper**, Outskirts 36.7 lakh cheaper, Suburb 27.2 lakh cheaper and Riverside 13.4 lakh cheaper, all else equal. **Location is therefore the second most important factor**, and each year of age lowers the price by about 0.53 lakh (10 years is about 5.3 lakh).
- Furnishing matters less: compared with fully furnished, semi-furnished is 4.0 lakh cheaper and unfurnished 7.0 lakh cheaper.
- Bedrooms add only +1.7 lakh each once area and bathrooms are known, which is a sign of the multicollinearity noted earlier.

## 12. Bonus: compare with Ridge and Lasso regularisation

In [ ]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=10)": Ridge(alpha=10),
    "Lasso (alpha=0.1)": Lasso(alpha=0.1, max_iter=10000),
}
rows = []
for name, m in reg_models.items():
    pipe = Pipeline([("scale", StandardScaler()), ("model", m)]).fit(Xtr, y_train)
    p = pipe.predict(Xte); mt = metrics(y_test, p)
    nz = int((pipe.named_steps["model"].coef_ != 0).sum())
    rows.append({"Model": name, "Test R2": mt["R2"], "Test RMSE": mt["RMSE"], "Non-zero coefficients": nz})
pd.DataFrame(rows).set_index("Model").round(3)

**Observation:** Ridge (R2 0.949, RMSE 7.11) and Lasso (R2 0.950, RMSE 7.10) perform essentially the same as plain Linear Regression (R2 0.951, RMSE 7.02), and Lasso keeps all 12 features (none shrunk to zero). Regularisation helps when there are many features or signs of overfitting; here there are only 12 features, plenty of data and no overfitting, so it adds nothing. **Plain Linear Regression is the best choice** because it is the simplest and the easiest to interpret.

## 13. Conclusion

- A simple **linear regression predicts house prices with R2 = 0.95 and a typical error of about 7 lakh (9%)**, and it generalises well (cross-validated R2 about 0.94).
- **Area is the most important driver, then location, then age.** Bathrooms, parking and main-road access add smaller premiums, and furnishing matters least.
- Residuals are random and centred, so the linear model fits this data well; Ridge and Lasso give no improvement.
- **Limitations:** the data is synthetic and was built with mostly linear relationships, so real housing data will give a lower R2 and show more non-linearity (for example, price per sqft varies by area). Next steps on real data: log-transform the price, add interaction terms (area x location), include more features, and compare with tree-based models such as Random Forest.